In [1]:
!git clone https://github.com/hyperchancellor07/NLP-Dialogue-Summarization.git

Cloning into 'NLP-Dialogue-Summarization'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 86 (delta 30), reused 68 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 14.17 MiB | 9.87 MiB/s, done.
Resolving deltas: 100% (30/30), done.
Updating files: 100% (58/58), done.


In [2]:
%cd NLP-Dialogue-Summarization

/content/NLP-Dialogue-Summarization


In [3]:
!pip install -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 125.0 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=fbbac97d6b0ecd6779e2463c7d677a4b69d35f802fd3e7cbf4c6ca02469a680b
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [4]:
!pip install -q transformers peft accelerate sentencepiece rouge_score datasets pandas

In [5]:
import torch
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)
from peft import (
    PeftModel,
)

In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Using device:", device)

Using device: cuda


In [7]:
dataset = load_dataset("knkarthick/samsum")
test_dataset = dataset["test"]
print(test_dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

{'id': '13862856', 'dialogue': "Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye", 'summary': "Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry."}


In [8]:
BASELINE_MODEL = "google/pegasus-large"
baseline_tokenizer = AutoTokenizer.from_pretrained(
    BASELINE_MODEL
)
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASELINE_MODEL
)
baseline_model.to(device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

...

Error during conversion: ReadTimeout('The read operation timed out')


model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-large
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

PegasusForConditionalGeneration(
  (model): PegasusModel(
    (shared): Embedding(96103, 1024, padding_idx=0)
    (encoder): PegasusEncoder(
      (embed_tokens): Embedding(96103, 1024, padding_idx=0)
      (embed_positions): PegasusSinusoidalPositionalEmbedding(1024, 1024)
      (layers): ModuleList(
        (0-15): 16 x PegasusEncoderLayer(
          (self_attn): PegasusAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (final_layer_no

In [9]:
FULL_FT_MODEL = "hyperchancellor07/pegasus-samsum-dialogue-summarizer"
full_ft_tokenizer = AutoTokenizer.from_pretrained(
    FULL_FT_MODEL
)
full_ft_model = AutoModelForSeq2SeqLM.from_pretrained(
    FULL_FT_MODEL
)
full_ft_model.to(device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

PegasusForConditionalGeneration(
  (model): PegasusModel(
    (shared): Embedding(96103, 1024, padding_idx=0)
    (encoder): PegasusEncoder(
      (embed_tokens): Embedding(96103, 1024, padding_idx=0)
      (embed_positions): PegasusSinusoidalPositionalEmbedding(1024, 1024)
      (layers): ModuleList(
        (0-15): 16 x PegasusEncoderLayer(
          (self_attn): PegasusAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (final_layer_no

In [11]:
!pip uninstall torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/torchao-0.10.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torchao/*
Proceed (Y/n)? y
  Successfully uninstalled torchao-0.10.0


In [12]:
LORA_MODEL = "hyperchancellor07/pegasus_lora"
lora_tokenizer = AutoTokenizer.from_pretrained(
    BASELINE_MODEL
)
base_lora_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASELINE_MODEL
)
lora_model = PeftModel.from_pretrained(
    base_lora_model,
    LORA_MODEL,
)
lora_model.to(device)

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

Error during conversion: ReadTimeout('The read operation timed out')
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-large
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider tra

adapter_model.safetensors:   0%|          | 0.00/12.6M [00:00<?, ?B/s]

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): PegasusForConditionalGeneration(
      (model): PegasusModel(
        (shared): Embedding(96103, 1024, padding_idx=0)
        (encoder): PegasusEncoder(
          (embed_tokens): Embedding(96103, 1024, padding_idx=0)
          (embed_positions): PegasusSinusoidalPositionalEmbedding(1024, 1024)
          (layers): ModuleList(
            (0-15): 16 x PegasusEncoderLayer(
              (self_attn): PegasusAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out_features=16, bias=False)
                  )
                  (lora_B): ModuleDict(


In [13]:
def generate_summary(
    model,
    tokenizer,
    dialogue_text,
):
    dialogue_text = "summarize: " + dialogue_text
    inputs = tokenizer(
        dialogue_text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }
    summary_ids = model.generate(
        **inputs,
        max_length=96,
        min_length=8,
        num_beams=8,
        repetition_penalty=2.5,
        no_repeat_ngram_size=3,
        length_penalty=1.0,
        early_stopping=True,
    )
    generated_summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True,
    )
    return generated_summary

In [14]:
sample = test_dataset[10]
dialogue = sample["dialogue"]
reference_summary = sample["summary"]
baseline_summary = generate_summary(
    baseline_model,
    baseline_tokenizer,
    dialogue,
)
full_ft_summary = generate_summary(
    full_ft_model,
    full_ft_tokenizer,
    dialogue,
)
lora_summary = generate_summary(
    lora_model,
    lora_tokenizer,
    dialogue,
)
print("=" * 80)
print("DIALOGUE")
print("=" * 80)
print(dialogue)
print("\nREFERENCE SUMMARY:\n")
print(reference_summary)
print("\nBASELINE PEGASUS SUMMARY:\n")
print(baseline_summary)
print("\nFULL FINETUNED SUMMARY:\n")
print(full_ft_summary)
print("\nLORA FINETUNED SUMMARY:\n")
print(lora_summary)

DIALOGUE
Wanda: Let's make a party!
Gina: Why?
Wanda: beacuse. I want some fun!
Gina: ok, what do u need?
Wanda: 1st I need too make a list
Gina: noted and then?
Wanda: well, could u take yours father car and go do groceries with me?
Gina: don't know if he'll agree
Wanda: I know, but u can ask :)
Gina: I'll try but theres no promisess
Wanda: I know, u r the best!
Gina: When u wanna go
Wanda: Friday?
Gina: ok, I'll ask

REFERENCE SUMMARY:

Wanda wants to throw a party. She asks Gina to borrow her father's car and go do groceries together. They set the date for Friday. 

BASELINE PEGASUS SUMMARY:

me Woburn institutions first – paid company Higuain part no process her 400 her need called know oncere usinging way); may want journey many simply may teacher know | well

FULL FINETUNED SUMMARY:

Gina will take Wanda's father car and go do groceries with her.

LORA FINETUNED SUMMARY:

Gina relaxation. know provided her It


In [15]:
comparison_df = pd.DataFrame({
    "Model": [
        "Baseline PEGASUS",
        "Full Fine-Tuned PEGASUS",
        "LoRA PEGASUS",
    ],

    "Generated Summary": [
        baseline_summary,
        full_ft_summary,
        lora_summary,
    ]
})
comparison_df

,Model,Generated Summary
0,Baseline PEGASUS,me Woburn institutions first – paid company Hi...
1,Full Fine-Tuned PEGASUS,Gina will take Wanda's father car and go do gr...
2,LoRA PEGASUS,Gina relaxation. know provided her It


In [16]:
real_world_dialogues = [

    """
    Sam: yoo u coming tmrw?
    Jay: maybe bro depends on attendance 💀
    Sam: teacher mad af today
    Jay: ikr 😭
    Sam: bring charger btw
    Jay: kk
    """,


    """
    Mike: training crashed again
    Alex: CUDA out of memory?
    Mike: yeah batch size too high
    Alex: switch to gradient accumulation
    Mike: trying LoRA now
    """,


    """
    Customer: my account stopped working
    Agent: suspicious login detected
    Customer: can u unlock it pls
    Agent: reset password after login
    """
]

In [17]:
for i, dialogue in enumerate(real_world_dialogues):

    print("\n" + "=" * 100)
    print(f"REAL-WORLD DRIFT SAMPLE {i+1}")
    print("=" * 100)
    print("\nDIALOGUE:\n")
    print(dialogue)
    baseline_summary = generate_summary(
        baseline_model,
        baseline_tokenizer,
        dialogue,
    )
    full_ft_summary = generate_summary(
        full_ft_model,
        full_ft_tokenizer,
        dialogue,
    )
    lora_summary = generate_summary(
        lora_model,
        lora_tokenizer,
        dialogue,
    )
    print("\nBASELINE SUMMARY:\n")
    print(baseline_summary)
    print("\nFULL FT SUMMARY:\n")
    print(full_ft_summary)
    print("\nLORA SUMMARY:\n")
    print(lora_summary)


REAL-WORLD DRIFT SAMPLE 1

DIALOGUE:


    Sam: yoo u coming tmrw?
    Jay: maybe bro depends on attendance 💀
    Sam: teacher mad af today
    Jay: ikr 😭
    Sam: bring charger btw
    Jay: kk
    

BASELINE SUMMARY:

Jay good these too through college through smiles through immigrants first 4 through UM should urgeve us Money through Course asset should stipulated completeve staffrun making no him week” – quickly Jayor well only mem ' day variety source Choir concern hours reasons fence well

FULL FT SUMMARY:

Jay is coming to Sam's class today.

LORA SUMMARY:

Sam It CSUo. join of

REAL-WORLD DRIFT SAMPLE 2

DIALOGUE:


    Mike: training crashed again
    Alex: CUDA out of memory?
    Mike: yeah batch size too high
    Alex: switch to gradient accumulation
    Mike: trying LoRA now
    

BASELINE SUMMARY:

mem flesh should Serviceve day Aug Children however through informed now socks leverage now socks 44 now socks

FULL FT SUMMARY:

Mike's training crashed again. Alex is trying L

In [18]:
print("=" * 80)
print("RESEARCH OBSERVATIONS")
print("=" * 80)

RESEARCH OBSERVATIONS


In [19]:
print(
    """
1. Baseline PEGASUS behavior:
- General summarization capability
- Weak conversational adaptation
- Possible hallucinations or repetition


2. Full Fine-Tuned PEGASUS behavior:
- Strong conversational understanding
- Better dialogue compression
- Improved robustness on SAMSum-like data


3. LoRA Fine-Tuned behavior:
- Parameter-efficient adaptation
- Reduced compute requirement
- Performance tradeoff vs full fine-tuning


4. Drift Analysis:
- Informal slang introduces distribution shift
- Emoji-heavy dialogue challenges semantic compression
- Technical jargon affects summarization consistency


5. Overall Research Insight:
- Transfer learning enables conversational adaptation
- Fine-tuning improves robustness under conversational drift
- PEFT methods provide efficient transformer adaptation
"""
)


1. Baseline PEGASUS behavior:
- General summarization capability
- Weak conversational adaptation
- Possible hallucinations or repetition


2. Full Fine-Tuned PEGASUS behavior:
- Strong conversational understanding
- Better dialogue compression
- Improved robustness on SAMSum-like data


3. LoRA Fine-Tuned behavior:
- Parameter-efficient adaptation
- Reduced compute requirement
- Performance tradeoff vs full fine-tuning


4. Drift Analysis:
- Informal slang introduces distribution shift
- Emoji-heavy dialogue challenges semantic compression
- Technical jargon affects summarization consistency


5. Overall Research Insight:
- Transfer learning enables conversational adaptation
- Fine-tuning improves robustness under conversational drift
- PEFT methods provide efficient transformer adaptation



In [21]:
!pip install -q evaluate rouge_score

In [22]:
import evaluate
rouge_metric = evaluate.load("rouge")

In [24]:
evaluation_samples = 50
baseline_predictions = []
full_ft_predictions = []
lora_predictions = []
reference_summaries = []

In [25]:
for i in range(evaluation_samples):
    sample = test_dataset[i]
    dialogue = sample["dialogue"]
    reference_summary = sample["summary"]
    baseline_summary = generate_summary(
        baseline_model,
        baseline_tokenizer,
        dialogue,
    )
    # FULL FT
    full_ft_summary = generate_summary(
        full_ft_model,
        full_ft_tokenizer,
        dialogue,
    )
    # LORA
    lora_summary = generate_summary(
        lora_model,
        lora_tokenizer,
        dialogue,
    )
    baseline_predictions.append(
        baseline_summary
    )
    full_ft_predictions.append(
        full_ft_summary
    )
    lora_predictions.append(
        lora_summary
    )
    reference_summaries.append(
        reference_summary
    )

In [27]:
baseline_rouge = rouge_metric.compute(
    predictions=baseline_predictions,
    references=reference_summaries,
)
full_ft_rouge = rouge_metric.compute(
    predictions=full_ft_predictions,
    references=reference_summaries,
)
lora_rouge = rouge_metric.compute(
    predictions=lora_predictions,
    references=reference_summaries,
)

In [28]:
results_df = pd.DataFrame({

    "Model": [
        "Baseline PEGASUS",
        "Full Fine-Tuned PEGASUS",
        "LoRA PEGASUS",
    ],

    "ROUGE-1": [
        baseline_rouge["rouge1"],
        full_ft_rouge["rouge1"],
        lora_rouge["rouge1"],
    ],

    "ROUGE-2": [
        baseline_rouge["rouge2"],
        full_ft_rouge["rouge2"],
        lora_rouge["rouge2"],
    ],

    "ROUGE-L": [
        baseline_rouge["rougeL"],
        full_ft_rouge["rougeL"],
        lora_rouge["rougeL"],
    ],
})

In [29]:
results_df

,Model,ROUGE-1,ROUGE-2,ROUGE-L
0,Baseline PEGASUS,0.013508,0.000000,0.012841
1,Full Fine-Tuned PEGASUS,0.433800,0.198466,0.359550
2,LoRA PEGASUS,0.166906,0.000000,0.135296


In [31]:
from pathlib import Path
RESULTS_DIR = Path(
    "/content/NLP-Dialogue-Summarization/artifacts/results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
results_df.to_csv(

    RESULTS_DIR / "rouge_results.csv",

    index=False,
)
print(
    "ROUGE results saved successfully."
)


ROUGE results saved successfully.


HYBRID drift robustness Score(HDRS)

In [32]:
!pip install -q sentence-transformers rank_bm25 scikit-learn

In [33]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

In [34]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [36]:
def semantic_similarity(
    reference,
    generated,
):
    reference_embedding = embedding_model.encode(
        [reference]
    )
    generated_embedding = embedding_model.encode(
        [generated]
    )
    similarity = cosine_similarity(
        reference_embedding,
        generated_embedding,
    )[0][0]
    return float(similarity)

In [37]:
def sparse_similarity(
    reference,
    generated,
):
    corpus = [reference.split()]
    bm25 = BM25Okapi(corpus)
    tokenized_generated = generated.split()
    score = bm25.get_scores(
        tokenized_generated
    )[0]
    normalized_score = score / 10
    return min(
        float(normalized_score),
        1.0,
    )

In [38]:
def compute_hdrs(
    reference_summary,
    generated_summary,
):
    semantic_score = semantic_similarity(
        reference_summary,
        generated_summary,
    )
    sparse_score = sparse_similarity(
        reference_summary,
        generated_summary,
    )
    hdrs = (
        0.7 * semantic_score
        +
        0.3 * sparse_score
    )
    return {
        "semantic_similarity": round(
            semantic_score,
            4,
        ),
        "sparse_similarity": round(
            sparse_score,
            4,
        ),
        "HDRS": round(
            hdrs,
            4,
        ),
    }

In [39]:
sample = test_dataset[15]
dialogue = sample["dialogue"]
reference_summary = sample["summary"]

In [40]:

baseline_summary = generate_summary(
    baseline_model,
    baseline_tokenizer,
    dialogue,
)
full_ft_summary = generate_summary(
    full_ft_model,
    full_ft_tokenizer,
    dialogue,
)
lora_summary = generate_summary(
    lora_model,
    lora_tokenizer,
    dialogue,
)

In [43]:
baseline_hdrs = compute_hdrs(
    reference_summary,
    baseline_summary,
)
full_ft_hdrs = compute_hdrs(
    reference_summary,
    full_ft_summary,
)
lora_hdrs = compute_hdrs(
    reference_summary,
    lora_summary,
)

In [46]:

hdrs_df = pd.DataFrame({
    "Model": [
        "Baseline PEGASUS",
        "Full Fine-Tuned PEGASUS",
        "LoRA PEGASUS",
    ],
    "Semantic Similarity": [
        baseline_hdrs["semantic_similarity"],
        full_ft_hdrs["semantic_similarity"],
        lora_hdrs["semantic_similarity"],
    ],
    "Sparse Similarity": [
        baseline_hdrs["sparse_similarity"],
        full_ft_hdrs["sparse_similarity"],
        lora_hdrs["sparse_similarity"],
    ],

    "HDRS": [
        baseline_hdrs["HDRS"],
        full_ft_hdrs["HDRS"],
        lora_hdrs["HDRS"],
    ]
})

In [47]:
hdrs_df

,Model,Semantic Similarity,Sparse Similarity,HDRS
0,Baseline PEGASUS,0.1657,0.0000,0.1160
1,Full Fine-Tuned PEGASUS,0.7574,-0.1609,0.4819
2,LoRA PEGASUS,0.3418,-0.0275,0.2310


In [48]:
hdrs_df.to_csv(
    RESULTS_DIR / "hdrs_results.csv",
    index=False,
)

In [49]:
%cd /content/NLP-Dialogue-Summarization

/content/NLP-Dialogue-Summarization


In [52]:
!find / -name "*.ipynb" 2>/dev/null

/usr/local/lib/python3.12/dist-packages/nbclassic/bundler/tests/resources/empty.ipynb
/usr/local/lib/python3.12/dist-packages/notebook/bundler/tests/resources/empty.ipynb
/usr/local/lib/python3.12/dist-packages/panel/tests/ui/io/app.ipynb
/usr/local/lib/python3.12/dist-packages/wasabi/tests/test-data/wasabi-test-notebook.ipynb
/usr/local/lib/python3.12/dist-packages/spreg/optional_imports.ipynb
/usr/local/lib/python3.12/dist-packages/holoviews/tests/ipython/notebooks/test_opts_image_cell_magic_offset.ipynb
/usr/local/lib/python3.12/dist-packages/holoviews/tests/ipython/notebooks/test_output_svg_line_magic.ipynb
/usr/local/lib/python3.12/dist-packages/holoviews/tests/ipython/notebooks/test_opts_image_line_magic.ipynb
/usr/local/lib/python3.12/dist-packages/holoviews/tests/ipython/notebooks/test_opts_image_cell_magic.ipynb
/usr/local/lib/python3.12/dist-packages/holoviews/examples/gallery/demos/plotly/trisurf3d_demo.ipynb
/usr/local/lib/python3.12/dist-packages/holoviews/examples/gallery

In [59]:
!find / -name "07_comparative_drift_analysis.ipynb" 2>/dev/null

In [60]:
!cp "/content/drive/MyDrive/Colab Notebooks/07_comparative_drift_analysis.ipynb" "/content/NLP-Dialogue-Summarization/notebooks/"

cp: cannot stat '/content/drive/MyDrive/Colab Notebooks/07_comparative_drift_analysis.ipynb': No such file or directory
